In [2]:
import subprocess
subprocess.run(['pip', 'uninstall', 'torchao', '-y', '-q'])
subprocess.run(['pip', 'install', 'peft==0.19.1', 'bitsandbytes', '-q'])

CompletedProcess(args=['pip', 'install', 'peft==0.19.1', 'bitsandbytes', '-q'], returncode=0)

In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))

GPU available: True
Device: Tesla T4
Compute capability: (7, 5)


In [3]:
import json, os, random, math, time, shutil
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    AutoModel, get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType
from torch.optim import AdamW

INPUT_DIR = "/kaggle/input"   # searches ALL attached datasets, no name needed
WORK_DIR  = "/kaggle/working/RL_project"

DATA_DIR    = f"{WORK_DIR}/data"
MODEL_DIR   = f"{WORK_DIR}/models"
REWARD_PATH = f"{MODEL_DIR}/reward_model"
RFT_PATH    = f"{MODEL_DIR}/rft_baseline"
CACHE_DIR   = f"{WORK_DIR}/rft_cache"

os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

def find_file(root_dir, filename):
    for root, dirs, files in os.walk(root_dir):
        if filename in files:
            return os.path.join(root, filename)
    return None

# --- Copy the 3 ESConv json files (they're at the top level) ---
for fname in ["esconv_train.json", "esconv_validation.json", "esconv_test.json"]:
    src = find_file(INPUT_DIR, fname)
    dst = os.path.join(DATA_DIR, fname)
    if src and not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f"Found and copied {fname}  <-  {src}")
    elif not src:
        print(f"WARNING: {fname} not found anywhere under {INPUT_DIR}")

# --- Find reward_model.pt wherever it actually is nested ---
reward_pt = find_file(INPUT_DIR, "reward_model.pt")
if reward_pt:
    src_dir = os.path.dirname(reward_pt)
    if not os.path.exists(REWARD_PATH):
        shutil.copytree(src_dir, REWARD_PATH)
    print(f"Found reward_model.pt at: {reward_pt}")
    print(f"Copied its folder to: {REWARD_PATH}")
else:
    print("WARNING: reward_model.pt not found anywhere under /kaggle/input")

# --- Sanity check before doing anything expensive ---
required = [
    os.path.join(DATA_DIR, "esconv_train.json"),
    os.path.join(DATA_DIR, "esconv_validation.json"),
    os.path.join(DATA_DIR, "esconv_test.json"),
    os.path.join(REWARD_PATH, "reward_model.pt"),
    os.path.join(REWARD_PATH, "config.json"),
]
missing = [p for p in required if not os.path.exists(p)]
if missing:
    print("\n!!! MISSING FILES — fix before continuing !!!")
    for m in missing:
        print(" -", m)
    raise SystemExit("Stopping: required files not found.")
else:
    print("\nAll required files found. Safe to continue.")

Found reward_model.pt at: /kaggle/input/datasets/shakibtaherjoy/rl-dataset/reward_model-20260812T080642Z-1-001/reward_model/reward_model.pt
Copied its folder to: /kaggle/working/RL_project/models/reward_model

All required files found. Safe to continue.


In [ ]:
import os
print(os.listdir("/kaggle/input"))

In [ ]:
for root, dirs, files in os.walk("/kaggle/input"):
    print(root, "->", files)

In [4]:
BASE_MODEL    = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
EPOCHS        = 3
BATCH_SIZE    = 4
GRAD_ACCUM    = 4
MAX_LENGTH    = 512
MAX_QUERY_LEN = 200
LEARNING_RATE = 2e-4
PATIENCE      = 2
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
LORA_TARGETS  = ["q_proj", "v_proj"]

N_PROMPTS    = 500
N_CANDIDATES = 3
TOP_K        = 2
SAVE_EVERY   = 25

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

SYSTEM_PROMPT = (
    "You are a supportive mental health counselor engaged in an "
    "emotional support conversation. Your role is to provide "
    "empathetic, evidence-based emotional support following "
    "structured support strategies. You must never make diagnostic "
    "claims, give medication advice, or validate harmful behaviors "
    "without therapeutic reframing. Always prioritize the person's "
    "safety and wellbeing."
)

class RewardModel(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(backbone, ignore_mismatched_sizes=True)
        self.head = nn.Linear(self.encoder.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.head(out.last_hidden_state[:, 0, :]).squeeze(-1)

def load_reward_model():
    print("  Loading reward model...")
    with open(f"{REWARD_PATH}/config.json") as f:
        cfg = json.load(f)
    backbone = cfg["backbone"]
    rm = RewardModel(backbone).float()
    sd = torch.load(f"{REWARD_PATH}/reward_model.pt", map_location="cpu", weights_only=True)
    rm.load_state_dict(sd)
    rm = rm.to(DEVICE).eval()
    tok = AutoTokenizer.from_pretrained(backbone)
    print(f"  Reward model loaded ({backbone})")
    return rm, tok

def score_response(rm, rm_tok, question, response):
    text = f"Question: {question} [SEP] Response: {response}"
    enc = rm_tok(text, max_length=256, padding="max_length", truncation=True, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        score = rm(enc["input_ids"], enc["attention_mask"]).item()
    return float(np.clip(score, -10, 10))

def build_prompt(triple):
    parts = []
    for t in triple["conversation_history"][-4:]:
        role = "Person" if t["role"] == "seeker" else "Counselor"
        parts.append(f"{role}: {t['text']}")
    context = " ".join(parts)
    return (f"<|system|> {SYSTEM_PROMPT}</s> "
            f"<|user|> [Strategy: {triple['strategy_label']}] "
            f"{context}</s> <|assistant|>")

def get_question(triple):
    seekers = [t for t in triple["conversation_history"] if t["role"] == "seeker"]
    return seekers[-1]["text"] if seekers else ""

def generate_candidates_with_recovery(model, tokenizer, rm, rm_tok, triples):
    cache_path = f"{CACHE_DIR}/rft_candidates.json"
    completed = {}
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            completed = json.load(f)
        print(f"  Resuming from {len(completed)} completed prompts...")
    else:
        print("  Starting fresh candidate generation...")

    print(f"  Total prompts to process: {len(triples)}")
    print(f"  Already done: {len(completed)}")
    print(f"  Remaining: {len(triples) - len(completed)}")
    est_min = (len(triples) - len(completed)) * N_CANDIDATES * 4 // 60
    print(f"  Estimated time remaining: ~{est_min} min")

    model.eval()
    start_time = time.time()

    for idx, triple in enumerate(triples):
        key = str(idx)
        if key in completed:
            continue

        prompt = build_prompt(triple)
        question = get_question(triple)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_QUERY_LEN).to(DEVICE)

        candidates = []
        for _ in range(N_CANDIDATES):
            try:
                with torch.no_grad():
                    out = model.generate(
                        **inputs, max_new_tokens=100, do_sample=True,
                        temperature=0.8, top_p=0.9,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id, use_cache=True,
                    )
                resp_ids = out[0][inputs["input_ids"].shape[1]:]
                decoded = tokenizer.decode(resp_ids, skip_special_tokens=True).strip()
                if decoded and len(decoded.split()) >= 3:
                    score = score_response(rm, rm_tok, question, decoded)
                    candidates.append({"response": decoded, "score": score})
            except Exception:
                continue

        if candidates:
            candidates.sort(key=lambda x: x["score"], reverse=True)
            completed[key] = {"prompt": prompt, "candidates": candidates[:TOP_K]}

        if (idx + 1) % SAVE_EVERY == 0:
            with open(cache_path, "w") as f:
                json.dump(completed, f, ensure_ascii=False)
            elapsed = (time.time() - start_time) / 60
            rate = (idx + 1) / elapsed if elapsed > 0 else 0
            remain = (len(triples) - idx - 1) / rate if rate > 0 else 0
            print(f"  Progress: {idx+1}/{len(triples)} | Done: {len(completed)} | "
                  f"Elapsed: {elapsed:.1f}min | ETA: {remain:.1f}min")

    with open(cache_path, "w") as f:
        json.dump(completed, f, ensure_ascii=False)

    rft_examples = []
    for key, data in completed.items():
        for cand in data["candidates"]:
            rft_examples.append({"prompt": data["prompt"], "response": cand["response"], "score": cand["score"]})

    print(f"\n  Candidate generation complete.")
    print(f"  Prompts processed: {len(completed)}")
    print(f"  RFT training examples: {len(rft_examples)}")
    return rft_examples

class RFTDataset(Dataset):
    def __init__(self, examples, tokenizer):
        self.examples = examples
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        prompt, response = ex["prompt"], ex["response"]
        full_text = prompt + " " + response + "</s>"
        prompt_ids = self.tokenizer.encode(prompt, add_special_tokens=False)
        full_ids = self.tokenizer.encode(full_text, add_special_tokens=False)
        if len(full_ids) > MAX_LENGTH:
            full_ids = full_ids[:MAX_LENGTH]
            prompt_ids = prompt_ids[:MAX_LENGTH]
        pad_id = self.tokenizer.pad_token_id
        pad_len = MAX_LENGTH - len(full_ids)
        input_ids = full_ids + [pad_id] * pad_len
        labels = ([-100] * min(len(prompt_ids), MAX_LENGTH) + full_ids[len(prompt_ids):] + [-100] * pad_len)[:MAX_LENGTH]
        attention_mask = ([1] * len(full_ids) + [0] * pad_len)[:MAX_LENGTH]
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

def compute_perplexity(model, val_loader):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            out = model(input_ids=ids, attention_mask=mask, labels=labels)
            valid = (labels != -100).sum().item()
            if valid > 0:
                total_loss += out.loss.item() * valid
                total_tokens += valid
    avg_loss = total_loss / max(total_tokens, 1)
    return math.exp(min(avg_loss, 20)), avg_loss

def train_rft():
    print("=" * 60)
    print("REJECTION SAMPLING FINE-TUNING — KAGGLE (path-fixed)")
    print("=" * 60)

    results_path = f"{MODEL_DIR}/rft_results.json"
    if os.path.exists(results_path):
        with open(results_path) as f:
            saved = json.load(f)
        if saved.get("final_perplexity", 999) < 50:
            print(f"Already trained. PPL: {saved['final_perplexity']:.2f}")
            return saved

    print("\n  Loading ESConv data...")
    with open(f"{DATA_DIR}/esconv_train.json") as f:
        train_triples = json.load(f)
    with open(f"{DATA_DIR}/esconv_validation.json") as f:
        val_triples = json.load(f)

    random.seed(42)
    sampled_triples = random.sample(train_triples, min(N_PROMPTS, len(train_triples)))
    print(f"  Sampled {len(sampled_triples)} prompts")

    print("\n  Loading TinyLlama...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="auto")
    model.config.use_cache = False

    lora_cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=LORA_TARGETS,
                           lora_dropout=LORA_DROPOUT, bias="none", task_type=TaskType.CAUSAL_LM)
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    print("\n  Loading reward model...")
    rm, rm_tok = load_reward_model()

    print("\n  PHASE 1: Candidate generation...")
    rft_examples = generate_candidates_with_recovery(model, tokenizer, rm, rm_tok, sampled_triples)

    if len(rft_examples) < 50:
        print("ERROR: Too few RFT examples.")
        return None

    del rm, rm_tok
    torch.cuda.empty_cache()

    print("\n  PHASE 2: Training on selected responses...")
    val_examples = []
    random.seed(99)
    val_sample = random.sample(val_triples, min(500, len(val_triples)))
    for triple in val_sample:
        val_examples.append({"prompt": build_prompt(triple), "response": triple["response"], "score": 0.0})

    train_ds = RFTDataset(rft_examples, tokenizer)
    val_ds = RFTDataset(val_examples, tokenizer)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable, lr=LEARNING_RATE, weight_decay=0.01)
    total_steps = (len(train_dl) // GRAD_ACCUM) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps)
    scaler = torch.cuda.amp.GradScaler()

    print(f"  Training on {len(rft_examples)} examples...")
    base_ppl, _ = compute_perplexity(model, val_dl)
    print(f"  Baseline PPL: {base_ppl:.2f}")

    best_ppl, best_epoch, epochs_no_improve = base_ppl, 0, 0

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(train_dl):
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            with torch.cuda.amp.autocast():
                out = model(input_ids=ids, attention_mask=mask, labels=labels)
                loss = out.loss / GRAD_ACCUM
            scaler.scale(loss).backward()
            if (step + 1) % GRAD_ACCUM == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
            total_loss += out.loss.item()
            if (step + 1) % 50 == 0:
                print(f"    Step {step+1}/{len(train_dl)} Loss: {total_loss/(step+1):.4f}")

        ppl, _ = compute_perplexity(model, val_dl)
        print(f"\n  Epoch {epoch+1}/{EPOCHS} PPL: {ppl:.2f}")
        if ppl < best_ppl:
            best_ppl, best_epoch, epochs_no_improve = ppl, epoch + 1, 0
            os.makedirs(RFT_PATH, exist_ok=True)
            model.save_pretrained(RFT_PATH)
            tokenizer.save_pretrained(RFT_PATH)
            print(f"  Best saved (PPL: {best_ppl:.2f})")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print("  Early stopping.")
                break

    print("\n  Generation quality test...")
    model.eval()
    with open(f"{DATA_DIR}/esconv_test.json") as f:
        test_triples = json.load(f)
    for triple in test_triples[:3]:
        prompt = build_prompt(triple)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_QUERY_LEN).to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7,
                                  pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id, use_cache=True)
        gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        seekers = [t for t in triple["conversation_history"] if t["role"] == "seeker"]
        print(f"  User: {seekers[-1]['text'][:80]}")
        print(f"  RFT:  {gen[:100]}\n")

    ppl_red = (base_ppl - best_ppl) / base_ppl * 100
    result = {
        "base_model": BASE_MODEL, "model_path": RFT_PATH,
        "training_method": "rejection_sampling_finetuning",
        "n_rft_examples": len(rft_examples), "n_candidates": N_CANDIDATES,
        "top_k": TOP_K, "best_epoch": best_epoch,
        "baseline_perplexity": base_ppl, "final_perplexity": best_ppl,
        "perplexity_reduction": f"{ppl_red:.1f}%",
    }
    with open(results_path, "w") as f:
        json.dump(result, f, indent=2)

    print(f"\n{'='*60}")
    print("WEEK 1 — RFT COMPLETE")
    print(f"Baseline PPL:  {base_ppl:.2f}")
    print(f"Final PPL:     {best_ppl:.2f}")
    print(f"Reduction:     {ppl_red:.1f}%")
    print("IMPORTANT: download rft_baseline/, rft_results.json, and")
    print("rft_cache/rft_candidates.json before your session ends —")
    print("/kaggle/working is wiped when the session closes.")
    print(f"{'='*60}")
    return result

result = train_rft()

Device: cuda
GPU:  Tesla T4
VRAM: 15.6 GB
REJECTION SAMPLING FINE-TUNING — KAGGLE (path-fixed)

  Loading ESConv data...


  Sampled 500 prompts

  Loading TinyLlama...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044

  Loading reward model...
  Loading reward model...


...

  Reward model loaded (microsoft/deberta-v3-base)

  PHASE 1: Candidate generation...
  Starting fresh candidate generation...
  Total prompts to process: 500
  Already done: 0
  Remaining: 500
  Estimated time remaining: ~100 min
  Progress: 25/500 | Done: 25 | Elapsed: 6.3min | ETA: 119.3min
  Progress: 50/500 | Done: 50 | Elapsed: 12.7min | ETA: 114.7min
  Progress: 75/500 | Done: 75 | Elapsed: 19.1min | ETA: 108.0min
  Progress: 100/500 | Done: 100 | Elapsed: 25.4min | ETA: 101.6min
  Progress: 125/500 | Done: 125 | Elapsed: 31.6min | ETA: 94.7min
  Progress: 150/500 | Done: 150 | Elapsed: 37.9min | ETA: 88.5min
  Progress: 175/500 | Done: 175 | Elapsed: 44.4min | ETA: 82.4min
  Progress: 200/500 | Done: 200 | Elapsed: 50.8min | ETA: 76.2min
  Progress: 225/500 | Done: 225 | Elapsed: 57.3min | ETA: 70.0min
  Progress: 250/500 | Done: 250 | Elapsed: 63.5min | ETA: 63.5min
  Progress: 275/500 | Done: 275 | Elapsed: 69.8min | ETA: 57.1min
  Progress: 300/500 | Done: 300 | Elapsed: 76.

/tmp/ipykernel_151/1500770440.py:261: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


  Baseline PPL: 27.43


/tmp/ipykernel_151/1500770440.py:277: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


    Step 50/250 Loss: 1.2410
    Step 100/250 Loss: 1.1986
    Step 150/250 Loss: 1.1869
    Step 200/250 Loss: 1.1588
    Step 250/250 Loss: 1.1427

  Epoch 1/3 PPL: 20.81
  Best saved (PPL: 20.81)
    Step 50/250 Loss: 1.0134
    Step 100/250 Loss: 1.0082
    Step 150/250 Loss: 1.0195
    Step 200/250 Loss: 1.0219
    Step 250/250 Loss: 1.0245

  Epoch 2/3 PPL: 22.35
    Step 50/250 Loss: 0.9777
    Step 100/250 Loss: 0.9838
    Step 150/250 Loss: 0.9944
    Step 200/250 Loss: 0.9916
    Step 250/250 Loss: 0.9872

  Epoch 3/3 PPL: 22.66
  Early stopping.

  Generation quality test...
  User: hi i am okay, a little bit sad though
  RFT:  Strategy: Question: Hi, I am okay, but I am a little bit sad. Can you provide me with some tips on h

  User: Well with the holidays coming up i have been very stressed and nervous about wha
  RFT:  [Strategy: Empathy and Understanding] Counselor: I can understand that. Do you want me to tell you h

  User: I have no choice but to go live at home over

In [5]:
import shutil
shutil.make_archive("/kaggle/working/rft_output", 'zip', "/kaggle/working/RL_project/models")
from IPython.display import FileLink
FileLink("/kaggle/working/rft_output.zip")

/kaggle/working/rft_output.zip

Cell 3
